# PTC Receiver Analysis
![alt text](ptc_thermal_model_radial.png)

Import packages

In [ ]:
import numpy as np
from CoolProp.CoolProp import PropsSI
from enn554 import sun
from enn554.math import cosd
from enn554.constants import σ
from scipy.optimize import minimize, Bounds
import matplotlib.pyplot as plt

## First way: from scratch

Parameters

In [ ]:
T_inf, T_sky = 22+273.15, 22-8+273.15 # both in K
U = 5 # m/s
mdot_htf = 10 # kg/s
η_0 = 0.81

τ_env = 0.97
ɛ_env = 0.86
α_env = 0.02

α_abs = 0.955
ɛ_abs = 0.14

HTF_name = 'INCOMP::TVP1'

Dia,Doa = 6.6e-2,7.0e-2 # m
Die, Doe = 10.9e-2,11.5e-2 # m
W = 4.8235 #m 

θ,DNI = 30,850
I_bn = DNI*W

Supporting functions

In [ ]:
IAM = cosd(θ) + 0.000884*θ - 0.00005369*θ**2

def nusselt_htf(T,pressure=1e7): # make high pressure since it's incompressible and it shouldn't matter too much
    T_htf,T_absorber,T_envelope = T[0],T[1],T[2]
    # Aai = np.pi/4*Dia**2
    k = PropsSI('conductivity','T',T_htf,'P',pressure,HTF_name)
    Pr = PropsSI('Prandtl','T',T_htf,'P',pressure,HTF_name)
    Prw = PropsSI('Prandtl','T',T_absorber,'P',pressure,HTF_name)
    μ = PropsSI('viscosity','T',T_htf,'P',pressure,HTF_name)
    ReD = 4*mdot_htf/(np.pi*Dia*μ)

    if ReD > 5e6:
        print('Warning: Reynolds number if outside the range of validity.')
    if ReD < 2300: # laminar
        Nu = 4.36
    else:
        f2 = (1.82*np.log10(ReD)-1.64)**(-2.0)
        Nu = f2/8.0 * (ReD-100)*Pr / (1+12.7*np.sqrt(f2/8.0) * (Pr**(2.0/3.0) - 1)) *(Pr/Prw)**0.11
    
    return Nu

def nusselt_envelope(T_envelope,T_air,wind_speed,D=None,air_pressure=101e3):
        
        T_film = 0.5*(T_envelope+T_air)
            
        mu = PropsSI('viscosity','T',T_film,'P',air_pressure,'Air')
        rho = PropsSI('D','T',T_film,'P',air_pressure,'Air')
        if wind_speed<0.1: # m/s
            Pr_film = PropsSI('Prandtl','T',T_film,'P',air_pressure,'Air')
            β = PropsSI('isobaric_expansion_coefficient','T',T_film,'P',air_pressure,'Air')
            k = PropsSI('conductivity','T',T_film,'P',air_pressure,'Air')
            cp = PropsSI('C','T',T_film,'P',air_pressure,'Air')
            α = k/rho/cp
            v = mu/rho            
            
            RaD = 9.81*β*Doe**3*(T_envelope-T_air)/(α*v)
            den = (1+ (0.559/Pr_film)**(9/16))**(16/9)
            Nu = (0.6 + 0.387*(RaD/den)**(1/6) )**2
        else:
            Pr_e = PropsSI('Prandtl','T',T_envelope,'P',air_pressure,'Air')
            Pr_ambient = PropsSI('Prandtl','T',T_air,'P',air_pressure,'Air')
            ReD = rho * wind_speed * Doe / mu

            if (Pr_ambient < 0.7) or (Pr_ambient>500):
                print('Warning: Prandtl number if outside the range of validity in the Nu computation.')
            if (ReD < 1.0) or (ReD > 1e6):
                print('Warning: Reynolds number if outside the range of validity in the Nu computation.')

            if ReD <= 40:
                C,m = 0.75,0.4
            elif ReD <= 1000:
                C,m = 0.51,0.5
            elif ReD<= 200e3:
                C,m = 0.26,0.6
            else:
                C,m = 0.076,0.7

            if Pr_ambient<=10:
                n=0.37
            else:
                n=0.36
            
            Nu = C*ReD**m * Pr_ambient**n * (Pr_ambient/Pr_e)**(1/4.0)
        
        return Nu   

def Q_ae_rad(T):
    T_htf,T_absorber,T_envelope = T[0],T[1],T[2]
    num = σ*np.pi*Doa*(T_absorber**4-T_envelope**4)
    den = 1/ɛ_abs + (1-ɛ_env)/ɛ_env * (Doa/Die)
    return num/den

Heat flows and residuals

In [ ]:
T = [400,610,310] # K
Q_a_abs = η_0*IAM*τ_env*α_abs * I_bn
Q_af_conv = np.pi * nusselt_htf(T) * PropsSI('conductivity','T',T[0],'P',1e7,HTF_name) * (T[1]-T[0])

Q_e_abs = η_0*IAM*α_env * I_bn
k_film = PropsSI('conductivity','T',0.5*(T_inf+T[2]),'P',101.3e3,'Air')
Q_sa_conv = np.pi*nusselt_envelope(T[-1],T_air=T_inf,wind_speed=U)*k_film*(T[2]-T_inf)
Q_es_rad = σ*Doe*np.pi*ɛ_env * (T[2]**4-T_sky**4)

print('============ absorber ==========')
print(f'Q_a_abs = {Q_a_abs:.2f}')
print(f'Q_af_conv = {Q_af_conv:.2f}')
print(f'Q_ae_rad = {Q_ae_rad(T):.2f}')

print('============ envelope ==========')
print(f'Q_e_abs = {Q_e_abs:.2f}')
print(f'Q_sa_conv = {Q_sa_conv:.2f}')
print(f'Q_es_rad = {Q_es_rad:.2f}')


In [ ]:
def residuals(T):
    Q_a_abs = η_0*IAM*τ_env*α_abs * I_bn
    Q_af_conv = np.pi * nusselt_htf(T) * PropsSI('conductivity','T',T[0],'P',1e7,HTF_name) * (T[1]-T[0])

    Q_e_abs = η_0*IAM*α_env * I_bn
    k_film = PropsSI('conductivity','T',0.5*(T_inf+T[2]),'P',101.3e3,'Air')
    Q_sa_conv = np.pi*nusselt_envelope(T[-1],T_air=T_inf,wind_speed=U)*k_film*(T[2]-T_inf)
    Q_es_rad = σ*Doe*np.pi*ɛ_env * (T[2]**4-T_sky**4)

    Q_ae_rad_val = Q_ae_rad(T)
    res_abs = Q_a_abs - Q_af_conv - Q_ae_rad_val
    res_env = Q_e_abs + Q_ae_rad_val - Q_sa_conv - Q_es_rad
    return res_abs**2 + res_env**2

Given HTF temperature, find absorber and envelop temperatures and compute the losses

In [ ]:
T_htf = 663
bnd = Bounds((286,286),(670,670), keep_feasible=True)
res = minimize(lambda x: residuals([T_htf,x[0],x[1]]),T[1::],bounds=bnd)

Ts = [T_htf,res.x[0],res.x[1]]
Q_af_conv = np.pi * nusselt_htf(Ts) * PropsSI('conductivity','T',Ts[0],'P',1e7,HTF_name) * (Ts[1]-Ts[0])
print(f'Heat gain={Q_af_conv:.3f}')
print(f'eta={Q_af_conv/DNI/W:.3f}')

In [ ]:
heat_gains = []
heat_losses = []
efficiency = []
temps = [100,150,200,250,300,350,390]
for T_htf in temps:
    T0 = [T_htf+273.15+10,T_inf]
    res = minimize(lambda x: residuals([T_htf+273.15,x[0],x[1]]),T0,bounds=bnd)
    
    Ts = [T_htf+273.15,res.x[0],res.x[1]]
    g = np.pi * nusselt_htf(Ts) * PropsSI('conductivity','T',Ts[0],'P',1e7,HTF_name) * (Ts[1]-Ts[0])
    k_film = PropsSI('conductivity','T',0.5*(T_inf+Ts[2]),'P',101.3e3,'Air')
    loss = np.pi*nusselt_envelope(Ts[-1],T_air=T_inf,wind_speed=U)*k_film*(Ts[2]-T_inf) + σ*Doe*np.pi*ɛ_env * (Ts[2]**4-T_sky**4)

    heat_gains.append(g)
    heat_losses.append(loss)
    efficiency.append(g/DNI/W)

fig,ax = plt.subplots(nrows=3)
ax[0].plot(temps,efficiency)
ax[0].set_ylabel('Efficiency')
ax[1].plot(temps,heat_gains)
ax[1].set_ylabel('Heat gains [W/m]')
ax[2].plot(temps,heat_losses)
ax[2].set_ylabel('Losses [W/m]')
fig.tight_layout()

## Second way: using toolbox